In [ ]:
# ---- evening sweep (Task 3 + one Task 1 refinement): what runs and until when ----------------------------------------
# runs next to evening_bayesian_fm_sweep.ipynb (CPU, one core) -> start both with Run All
# every sweep has a time budget (minutes): no new run once its budget is used up or the overall end is reached
# (a run that has started always finishes); the grids run in random order -> a cut sweep still covers its range
END_TIME = "23:59"                              # overall end (today, clock of this PC); "end_at" in evening_queue.json can move it
run_task3_fast_models = True                    # time-aware bias model, temporal zero model, time-decayed kNN   budgets ~0.7 h
run_task3_time_factorization_machine = True     # the 3 stages of task3_sweep.ipynb that never ran              ~0.9 h
run_task3_time_svdpp = True                     # factor penalty x learning rate, then the new temporal biases  ~0.5 h
run_task3_new_models = True                     # first Task 3 runs: integrated model, AutoRec, transformer      ~0.9 h
run_task1_autorec = True                        # Task 1 AutoRec below its best learning rate (grid edge)        ~0.25 h
run_queue = True                                # then the refinement sweeps in evening_queue.json, until the end

# Evening sweep: Task 3 (+ one Task 1 refinement)

**Run All** once, together with `evening_bayesian_fm_sweep.ipynb` (the Bayesian FMs run on one CPU core, this notebook mostly on
the GPU). Every sweep cell only runs if its flag is `True`; the notebook stops at `END_TIME`.

| part | models | time per run (estimate) |
|---|---|---|
| Task 3 fast | time-aware bias model (closed form), temporal zero model (log loss of P(0)), time-decayed item kNN (CPU) | seconds - 1 min |
| Task 3 time-aware FM | learning rate x batch, factors x penalty, bias lambda (calendar fields month + year, the newer temporal features) | 2 - 6 min |
| Task 3 timeSVD++ | factor penalty x learning rate, then with the temporal biases found above (+ drifting user factors) | ~3 - 5 min |
| Task 3 new models | integrated model, I-AutoRec, rating transformer (the Task 1 winners as starting points) | 5 - 15 min |
| Task 1 AutoRec | learning rate 1.5e-5 (3e-5 was the best and the edge of the grid) | ~10 min |
| refinements | the sweeps in `evening_queue.json` (written while the notebook runs) | |

Same mechanics as `overnight_sweep.ipynb`: time budgets per sweep, random order, a failing sweep is logged and skipped, the computer
is kept awake, no autoreload (the code and `lib/config.py` as they were at the start). Staged sweeps pass their best values on.
Scores: validation MSE (lower is better), log loss of P(0) for the zero model. Results: `output/<task>/<model>_<start time>.json`,
one line per sweep in `output/evening_log.txt`, summary table in the last cell.

In [ ]:
import json
import time
import traceback
from datetime import datetime, timedelta

import torch

from grid_search import OUTPUT_DIR, SWEEP_DIR, best_values_of, keep_computer_awake, print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

QUEUE_PATH = SWEEP_DIR / "evening_queue.json"
LOG_PATH = OUTPUT_DIR / "evening_log.txt"
NO_RESULT = {"best": None}                      # what a skipped or failed sweep returns (best_values_of -> {})

sweep_start = datetime.now()
end_hour, end_minute = map(int, END_TIME.split(":"))
overall_end = sweep_start.replace(hour=end_hour, minute=end_minute, second=0, microsecond=0)
if overall_end <= sweep_start:                  # END_TIME already passed today -> tomorrow
    overall_end += timedelta(days=1)
sweep_results = {}                              # sweep label -> report (summary in the last cell)


def write_log(line: str):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(f"{datetime.now():%Y-%m-%d %H:%M:%S} {line}\n")


# one sweep with a time budget: no new run after budget_minutes or at the overall end, combinations in random order
# (a cut sweep still covers the whole grid); a failing sweep is printed and skipped, the next one starts
def run_timeboxed_sweep(label: str, task_name: str, model_name: str, parameter_grid: dict, budget_minutes: float, **options) -> dict:
    now = datetime.now()
    if now >= overall_end - timedelta(minutes=1):
        print(f"\nskipped {label}: overall end {overall_end:%H:%M} reached", flush=True)
        write_log(f"skipped  {label} (overall end reached)")
        return NO_RESULT
    deadline = min(now + timedelta(minutes=budget_minutes), overall_end)
    print(f"\n===== {label} | budget {budget_minutes:.0f} min -> no new run after {deadline:%H:%M} =====", flush=True)
    write_log(f"start    {label} | {task_name} / {model_name} | until {deadline:%H:%M}")
    try:
        report = run_grid_search(task_name, model_name, parameter_grid, deadline=deadline, random_order_seed=0, **options)
    except Exception as error:
        traceback.print_exc()
        print(f"{label} failed -> continuing with the next sweep", flush=True)
        write_log(f"FAILED   {label}: {type(error).__name__}: {error}"[:400])
        return NO_RESULT
    finally:
        keep_computer_awake(True)               # run_grid_search lets the computer sleep again at its end -> hold it until the end

    sweep_results[label] = report
    output_name = f"{task_name}/{model_name}_{report['started'].replace(' ', '_').replace(':', '-')}.json"
    best = report["best"]
    write_log(f"finished {label} | {report['status']} | {report['runs_finished']} runs | "
              + (f"best {best['score']:.5f} with {json.dumps(best['swept_values'])}" if best else "no run finished")
              + f" | {output_name}")
    return report


# best values of a time-aware bias model sweep -> the same settings inside timeSVD++ (its config nests the time bias model)
def as_time_svdpp_biases(time_bias_model_values: dict) -> dict:
    return {f"time_biases.{name}": value for name, value in time_bias_model_values.items()}


keep_computer_awake(True)
write_log(f"sweep started, overall end {overall_end:%Y-%m-%d %H:%M}")
print(f"start {sweep_start:%H:%M} | overall end {overall_end:%Y-%m-%d %H:%M} | log: {LOG_PATH}")

## Task 3 - fast models: wide grids
### Time-aware bias model (closed form; the temporal biases are shared with timeSVD++)
Config: 30 time bins, user periods 1 / 7 / 30 / 90 / 365 days, drift exponent 0.4, lambdas 50 (time biases) / 3000 (drift) / 3 (b_u, b_i).

In [ ]:
if run_task3_fast_models:
    # stage 1 - shrinkage of the time biases x of the drift x drift exponent (27 runs)
    time_bias_shrinkage_sweep = run_timeboxed_sweep("T3 time bias 1 shrinkage / drift", "task3", "time_bias_model", {
        "temporal_biases.time_bias_regularization": [25.0, 50.0, 100.0],
        "temporal_biases.drift_regularization": [1000.0, 3000.0, 10000.0],
        "temporal_biases.drift_exponent": [0.2, 0.4, 0.6],
    }, budget_minutes=8)
    time_bias_best_values = best_values_of(time_bias_shrinkage_sweep)

    # stage 2 - time bins x b_u / b_i lambda x user periods (27 runs + the best 3 on split seeds 1 / 2)
    time_bias_structure_sweep = run_timeboxed_sweep("T3 time bias 2 bins / periods", "task3", "time_bias_model", {
        **time_bias_best_values,
        "temporal_biases.n_time_bins": [10, 30, 60],
        "bias_regularization": [2.0, 3.0, 5.0],
        "temporal_biases.user_period_lengths_in_days": [(1, 7, 30, 90, 365), (1, 3, 7, 14, 30, 90, 365), (1, 7, 30, 90, 365, 1000)],
    }, budget_minutes=8, rerun_finalists_on_other_splits=True, n_finalists=3)
    time_bias_best_values |= best_values_of(time_bias_structure_sweep)

### Temporal zero model (log loss of P(0)) and time-decayed item kNN (CPU)

In [ ]:
if run_task3_fast_models:
    # zero model: which windows around the day count x their shrinkage (16 runs)
    time_zero_model_sweep = run_timeboxed_sweep("T3 time zero model", "task3", "time_zero_model", {
        "window_lengths_in_days": [(0, 30), (0, 7, 30), (0, 30, 365), (0, 7, 30, 365)],
        "window_prior_strength": [0.5, 1.0, 2.0, 5.0],
    }, budget_minutes=5)

    # time-decayed kNN: decay length x floor x neighbours (24 runs, about a minute each)
    time_item_knn_sweep = run_timeboxed_sweep("T3 time-decayed kNN", "task3", "time_item_knn", {
        "time_decay_days": [90.0, 365.0, 1000.0, 3000.0],
        "time_decay_floor": [0.3, 0.5, 0.7],
        "n_neighbors": [50, 100],
    }, budget_minutes=20)

## Task 3 - time-aware factorization machine (GPU; calendar fields month + year)
The stages 2 - 4 of `task3_sweep.ipynb` (stage 1, the calendar fields, ran on 14 Nov: month + year 0.89413).

In [ ]:
if run_task3_time_factorization_machine:
    # stage 2 - learning rate x batch size (6 runs; Task 1: 5e-4 / 256 was best)
    time_fm_optimisation_sweep = run_timeboxed_sweep("T3 time FM 2 learning rate / batch", "task3", "time_factorization_machine", {
        "learning_rate": [2.5e-4, 5e-4, 1e-3],
        "ratings_per_batch": [256, 512],
    }, budget_minutes=18)
    time_fm_best_values = best_values_of(time_fm_optimisation_sweep)

    # stage 3 - factors x factor penalty (6 runs; time ~ proportional to k)
    time_fm_capacity_sweep = run_timeboxed_sweep("T3 time FM 3 factors / penalty", "task3", "time_factorization_machine", {
        **time_fm_best_values,
        "n_factors": [120, 250],
        "factor_regularization": [8.0, 16.0, 32.0],
    }, budget_minutes=25)
    time_fm_best_values |= best_values_of(time_fm_capacity_sweep)

    # stage 4 - one lambda for b_u and b_i (3 runs)
    time_fm_bias_sweep = run_timeboxed_sweep("T3 time FM 4 bias lambda", "task3", "time_factorization_machine", {
        **time_fm_best_values,
        "bias_regularization": [1.0, 3.0, 5.0],
    }, budget_minutes=8)

## Task 3 - timeSVD++ (GPU; SVD++ on top of the closed-form time-aware bias model)

In [ ]:
if run_task3_time_svdpp:
    # stage 1 - factor penalty x learning rate (6 runs)
    time_svdpp_sweep = run_timeboxed_sweep("T3 timeSVD++ 1 penalty / learning rate", "task3", "time_svdpp", {
        "factor_regularization": [4.0, 8.0, 16.0],
        "learning_rate": [1e-3, 2e-3],
    }, budget_minutes=20)
    time_svdpp_best_values = best_values_of(time_svdpp_sweep)

    # stage 2 - the temporal biases found above (if the bias sweep ran) x drifting user factors p_u(t) (2 runs)
    time_svdpp_bias_sweep = run_timeboxed_sweep("T3 timeSVD++ 2 new biases / factor drift", "task3", "time_svdpp", {
        **time_svdpp_best_values,
        **(as_time_svdpp_biases(time_bias_best_values) if run_task3_fast_models else {}),
        "use_factor_drift": [False, True],
    }, budget_minutes=12)

## Task 3 - the new models (first runs on Task 3; starting points = the Task 1 sweep winners)

In [ ]:
if run_task3_new_models:
    # integrated model: factor penalty 24 (Task 1 best) and 16 (2 runs, ~10 min each)
    task3_integrated_model_sweep = run_timeboxed_sweep("T3 integrated model", "task3", "integrated_model", {
        "factor_regularization": [16.0, 24.0],
    }, budget_minutes=25)

    # I-AutoRec: hidden 250 (Task 1 best) and 500 (2 runs)
    task3_autorec_sweep = run_timeboxed_sweep("T3 AutoRec", "task3", "autorec", {
        "hidden_dimension": [250, 500],
    }, budget_minutes=15)

    # rating transformer: the Task 1 settings (1 run: its score on Task 3)
    task3_rating_transformer_sweep = run_timeboxed_sweep("T3 rating transformer", "task3", "rating_transformer", {},
                                                         budget_minutes=15)

## Task 1 - AutoRec below its best learning rate (3e-5 was the edge of the grid: 0.7490)

In [ ]:
if run_task1_autorec:
    task1_autorec_sweep = run_timeboxed_sweep("T1 AutoRec learning rate 1.5e-5", "task1", "autorec", {
        "learning_rate": 1.5e-5,
        "hidden_dimension": 250,
        "regularization": 16.0,
        "masked_fraction": 0.25,
    }, budget_minutes=15)

## Refinement queue
Runs every entry of the queue file it has not run yet (by `name`), re-reading the file every minute, until the overall end.

In [ ]:
def read_queue() -> dict:
    try:
        return json.loads(QUEUE_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        print(f"queue file not readable ({type(error).__name__}: {error}) -> retry in a minute", flush=True)
        return {"sweeps": []}


# JSON has no tuples: a list inside a list ([[...], [...]]) = values to sweep, each one a tuple setting (e.g. calendar fields)
def grid_from_json(grid: dict) -> dict:
    return {name: ([tuple(value) if isinstance(value, list) else value for value in values]
                   if isinstance(values, list) and any(isinstance(value, list) for value in values) else values)
            for name, values in grid.items()}


if run_queue:
    finished_queue_entries = set()
    while True:
        queue = read_queue()
        if queue.get("end_at"):                 # the queue file can move the overall end
            overall_end = datetime.strptime(queue["end_at"], "%Y-%m-%d %H:%M")
        if datetime.now() >= overall_end - timedelta(minutes=1):
            print(f"overall end {overall_end:%H:%M} reached -> queue stopped", flush=True)
            break
        pending_entries = [entry for entry in queue.get("sweeps", []) if entry["name"] not in finished_queue_entries]
        if not pending_entries:
            time.sleep(60)
            continue
        entry = pending_entries[0]
        finished_queue_entries.add(entry["name"])
        run_timeboxed_sweep(f"queue: {entry['name']}", entry["task"], entry["model"], grid_from_json(entry["grid"]),
                            entry.get("budget_minutes", 20), **entry.get("options", {}))

## Summary
One line per sweep of this notebook (also in the log file), then every sweep file of the tasks.

In [ ]:
for label, report in sweep_results.items():
    best = report["best"]
    best_text = f"{best['score']:.5f} with {best['swept_values']}" if best else "no finished run"
    print(f"{label:<44} {report['status']:<24} {report['runs_finished']:>9} | best {best_text}")

for task_name in ('task3', 'task1'):
    print(f"\n---- {task_name} ----")
    print_sweep_overview(task_name)

keep_computer_awake(False)
write_log("sweep finished")